In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.linear_model import LinearRegression
from sklearn.base import BaseEstimator,TransformerMixin

In [2]:
class IQROutlierCapper(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        self.factor = factor
        self.lower_bounds_ = {}
        self.upper_bounds_ = {}

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        
        q25 = X_df.quantile(0.25)
        q75 = X_df.quantile(0.75)
        iqr = q75 - q25
        
        for col in X_df.columns:
            self.lower_bounds_[col] = q25[col] - (self.factor * iqr[col])
            self.upper_bounds_[col] = q75[col] + (self.factor * iqr[col])
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        
        for col in X_df.columns:
            if col in self.lower_bounds_:
                X_df[col] = np.clip(X_df[col], self.lower_bounds_[col], self.upper_bounds_[col])
        
        return X_df.to_numpy() if isinstance(X, np.ndarray) else X_df

In [3]:
housing =pd.read_csv("Housing.csv")

In [4]:
x_train,x_test,y_train,y_test=train_test_split(housing.drop(columns=['price']),housing['price'],test_size=0.25,random_state=42)

In [5]:
t1=ColumnTransformer([
    ('impute_bathrooms',SimpleImputer(strategy='most_frequent'),[2]),
    ('impute_mainroad',SimpleImputer(strategy='most_frequent'),[4])
],remainder='passthrough')

In [6]:
t2=ColumnTransformer([
    ('encode_mr_gr_bm_hw_ac_pa',OrdinalEncoder(categories=[['no','yes']]*6),[1,5,6,7,8,10]),
    ('encode_fr',OrdinalEncoder(categories=[['unfurnished','semi-furnished','furnished']]),[11])
],remainder='passthrough')

In [7]:
t3=ColumnTransformer([
    ('outlier',IQROutlierCapper(),[8])
],remainder='passthrough')

In [8]:
t5=LinearRegression()

In [9]:
pipe=Pipeline([
    ('t1',t1),
    ('t2',t2),
    ('t3',t3),
    ('t5',t5)
])

In [10]:
pipe.fit(x_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('t1', ...), ('t2', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](12,)","['area','bedrooms','bathrooms',...,'parking','prefarea','furnishingstatus']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('impute_bathrooms', ...), ('impute_mainroad', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. T

In [11]:
y_pred=pipe.predict(x_test)

In [12]:
from sklearn.metrics import r2_score ,mean_absolute_error,mean_squared_error
r2=r2_score(y_test,y_pred)
mae=mean_absolute_error(y_test,y_pred)
mse=mean_squared_error(y_test,y_pred)

In [13]:
(r2,mae,mse)

(0.6719052057615904, 904896.9517834972, 1511941912316.2012)

In [14]:
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))